In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

import torch
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torchmetrics as tm
from transformers import get_linear_schedule_with_warmup

In [2]:
root_path = "/home/stefan/ioai-prep/kits/autor_versuri"
device = "cuda" if torch.cuda.is_available() else "cpu"

seed = 423
torch.random.manual_seed(seed)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import VotingClassifier, StackingClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.base import BaseEstimator, TransformerMixin
from scipy.sparse import hstack
import re
import os

root_path = "/home/stefan/ioai-prep/kits/autor_versuri"

train_df = pd.read_csv(os.path.join(root_path, "train.csv"))
test_df = pd.read_csv(os.path.join(root_path, "test.csv"))

le = LabelEncoder()
y = le.fit_transform(train_df["Autor"])
X_train_text = train_df["Versuri"].values
X_test_text = test_df["Versuri"].values


# === Stylometric Features ===
class StyleFeatures(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        features = []
        for text in X:
            lines = text.strip().split("\n")
            words = re.findall(r"\b\w+\b", text.lower())
            chars = re.sub(r"\s", "", text)

            f = {
                "avg_word_len": np.mean([len(w) for w in words]) if words else 0,
                "avg_line_len": np.mean([len(l) for l in lines]) if lines else 0,
                "num_lines": len(lines),
                "vocab_richness": len(set(words)) / len(words) if words else 0,
                "punct_ratio": sum(1 for c in text if c in ".,;:!?-—") / len(text),
                "comma_ratio": text.count(",") / len(text),
                "ellipsis": text.count("...") / len(text) * 100,
                "exclaim_ratio": text.count("!") / len(text),
                "question_ratio": text.count("?") / len(text),
                "dash_ratio": (text.count("-") + text.count("—")) / len(text),
                "uppercase_ratio": (
                    sum(1 for c in chars if c.isupper()) / len(chars) if chars else 0
                ),
                "digit_ratio": sum(1 for c in text if c.isdigit()) / len(text),
                # Romanian-specific patterns
                "diacritics": sum(1 for c in text if c in "ăâîșțĂÂÎȘȚ") / len(text),
                # Line ending patterns (rhyme hints)
                "lines_ending_a": sum(
                    1 for l in lines if l.strip().lower().endswith("a")
                ),
                "lines_ending_e": sum(
                    1 for l in lines if l.strip().lower().endswith("e")
                ),
                "lines_ending_i": sum(
                    1 for l in lines if l.strip().lower().endswith("i")
                ),
            }
            features.append(list(f.values()))
        return np.array(features)


# === Character n-grams (key for authorship) ===
char_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 5),
    max_features=15000,
    sublinear_tf=True,
)

# === Word n-grams ===
word_vectorizer = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    max_features=10000,
    sublinear_tf=True,
    min_df=2,
)


# === Character n-grams on line endings (captures rhyme patterns) ===
def extract_line_endings(texts, n_chars=8):
    result = []
    for text in texts:
        lines = text.strip().split("\n")
        endings = " ".join(
            [l.strip()[-n_chars:] for l in lines if len(l.strip()) >= n_chars]
        )
        result.append(endings)
    return result


line_end_train = extract_line_endings(X_train_text)
line_end_test = extract_line_endings(X_test_text)

ending_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 4),
    max_features=3000,
)

# === Build features ===
print("Building features...")
X_char = char_vectorizer.fit_transform(X_train_text)
X_word = word_vectorizer.fit_transform(X_train_text)
X_endings = ending_vectorizer.fit_transform(line_end_train)
X_style = StyleFeatures().fit_transform(X_train_text)

scaler = StandardScaler()
X_style_scaled = scaler.fit_transform(X_style)

X_combined = hstack([X_char, X_word, X_endings, X_style_scaled])

X_char_test = char_vectorizer.transform(X_test_text)
X_word_test = word_vectorizer.transform(X_test_text)
X_endings_test = ending_vectorizer.transform(line_end_test)
X_style_test = scaler.transform(StyleFeatures().transform(X_test_text))

X_test_combined = hstack([X_char_test, X_word_test, X_endings_test, X_style_test])

print(f"Feature dimensions: {X_combined.shape}")

# === Models ===
models = [
    ("lr", LogisticRegression(C=5, max_iter=2000, solver="saga", n_jobs=-1)),
    ("svc", LinearSVC(C=0.5, max_iter=7000, dual=True)),
    # (
    #     "lr_l1",
    #     LogisticRegression(C=3, max_iter=5000, solver="saga", penalty="l1", n_jobs=-1),
    # ),
]

# === Cross-validation ===
print("\nCross-validation scores:")
for name, model in models:
    scores = cross_val_score(model, X_combined, y, cv=3, scoring="accuracy", n_jobs=-1)
    print(f"{name}: {scores.mean():.4f} (+/- {scores.std()*2:.4f})")

# === Stacking Ensemble ===
stacking = StackingClassifier(
    estimators=models,
    final_estimator=LogisticRegression(C=1, max_iter=1000),
    cv=3,
    n_jobs=-1,
)

print("\nStacking CV score:")
stack_scores = cross_val_score(stacking, X_combined, y, cv=5, scoring="accuracy")
print(f"Stacking: {stack_scores.mean():.4f} (+/- {stack_scores.std()*2:.4f})")

# === Train and predict ===
stacking.fit(X_combined, y)
preds = stacking.predict(X_test_combined)

submission = pd.DataFrame(
    {
        "Id": test_df["Id"],
        "Autor": le.inverse_transform(preds),
    }
)
submission.to_csv(os.path.join(root_path, "submission.csv"), index=False)
print("\nSubmission saved.")

Building features...
Feature dimensions: (3415, 28016)

Cross-validation scores:
lr: 0.7283 (+/- 0.0058)


/home/stefan/.local/lib/python3.13/site-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/stefan/.local/lib/python3.13/site-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


svc: 0.7628 (+/- 0.0049)


KeyboardInterrupt: 

# Data

In [3]:
def clean_text(text):
    text = text.replace("ţ", "ț").replace("ş", "ș").replace("Ţ", "Ț").replace("Ş", "Ș")
    return text

In [4]:
def prep_df(df: pd.DataFrame):
    df = df[["Versuri"]].copy()
    df["Versuri"] = df["Versuri"].apply(clean_text)
    return df

In [5]:
df = pd.read_csv(f"{root_path}/train.csv")
le = LabelEncoder()
y = le.fit_transform(df["Autor"])

train_df = prep_df(df)

X_train, X_val, y_train, y_val = train_test_split(
    train_df["Versuri"].values, y, test_size=0.01, random_state=seed, stratify=y
)

train_df.head()

,Versuri
0,"Pe barbari de-i risipește,\nȘ-apoi vecinic pri..."
1,Sau va avea mulți copii.\nPentru că totuna est...
2,"Copilăriei noastre.\nFrate, ei urăsc\nCântecul..."
3,"Veniți lângă mine, tovarăși! Că mâne-o să mor,..."
4,"Iar tu, Hyperion, rămâi\nOriunde ai apune...\n..."


In [6]:
len(df)

3415

# Dataset

In [7]:
class VersuriDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "labels": torch.tensor(label, dtype=torch.long),
        }

In [8]:
model_name = "xlm-roberta-base"  # "dumitrescustefan/bert-base-romanian-uncased-v1"
tokenizer = AutoTokenizer.from_pretrained(model_name)

train_dataset = VersuriDataset(X_train, y_train, tokenizer)
val_dataset = VersuriDataset(X_val, y_val, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, pin_memory=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, pin_memory=True, num_workers=4)

# Model

In [9]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=len(le.classes_)
)
model.dropout = torch.nn.Dropout(0.3)
model.to(device)

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


XLMRobertaForSequenceClassification(
  (roberta): XLMRobertaModel(
    (embeddings): XLMRobertaEmbeddings(
      (word_embeddings): Embedding(250002, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): XLMRobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x XLMRobertaLayer(
          (attention): XLMRobertaAttention(
            (self): XLMRobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): XLMRobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=

# Train

In [10]:
num_epochs = 5

optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=1e-2)
accuracy_metric = tm.Accuracy(task="multiclass", num_classes=len(le.classes_)).to(device)
scaler = torch.amp.GradScaler(device)
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=0, num_training_steps=num_epochs * len(train_loader)
)

In [11]:
def train_epoch(model, loader, optimizer, device, metric):
    model.train()
    total_loss = 0
    metric.reset()

    for batch in tqdm(loader, desc="Training"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        with torch.amp.autocast(device):
            outputs = model(
                input_ids=input_ids, attention_mask=attention_mask, labels=labels
            )
            loss = outputs.loss

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer),
        scaler.update()
        scheduler.step()

        total_loss += loss.item()
        metric.update(outputs.logits.argmax(dim=-1), labels)

    avg_loss = total_loss / len(loader)
    acc = metric.compute()

    return avg_loss, acc

In [12]:
def evaluate(model, loader, device, metric):
    model.eval()
    total_loss = 0
    metric.reset()

    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids, attention_mask=attention_mask, labels=labels
            )

            loss = outputs.loss
            total_loss += loss.item()
            metric.update(outputs.logits.argmax(dim=-1), labels)

    avg_loss = total_loss / len(loader)
    acc = metric.compute()

    return avg_loss, acc

In [13]:
for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(
        model, train_loader, optimizer, device, accuracy_metric
    )
    val_loss, val_acc = evaluate(model, val_loader, device, accuracy_metric)

    print(f"Epoch {epoch + 1}/{num_epochs}")
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
    print()

    if epoch+1 ==4:
        break

Evaluating: 100%|██████████| 2/2 [00:00<00:00,  3.43it/s]


Epoch 1/5
Train Loss: 1.6997, Train Acc: 0.3648
Val Loss: 1.3663, Val Acc: 0.4571



Evaluating: 100%|██████████| 2/2 [00:00<00:00,  3.66it/s]


Epoch 2/5
Train Loss: 1.3034, Train Acc: 0.5293
Val Loss: 0.9931, Val Acc: 0.6571



Evaluating: 100%|██████████| 2/2 [00:00<00:00,  3.55it/s]


Epoch 3/5
Train Loss: 0.9520, Train Acc: 0.6669
Val Loss: 1.2337, Val Acc: 0.7143



Evaluating: 100%|██████████| 2/2 [00:00<00:00,  3.57it/s]

Epoch 4/5
Train Loss: 0.7253, Train Acc: 0.7577
Val Loss: 1.0799, Val Acc: 0.7714



# Submission

In [14]:
df_test_orig = pd.read_csv(f"{root_path}/test.csv")
df_test = prep_df(df_test_orig)

test_dataset = VersuriDataset(
    df_test["Versuri"].values, np.zeros(len(df_test)), tokenizer
)

test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

In [15]:
def predict(model, loader, device):
    model.eval()
    predictions = []

    with torch.no_grad():
        for batch in tqdm(loader, desc="Predicting"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = outputs.logits.argmax(dim=-1)
            predictions.extend(preds.cpu().numpy())

    return predictions

In [16]:
predictions = predict(model, test_loader, device)

submission = df_test_orig[["Id"]].copy()
submission["Autor"] = le.inverse_transform(predictions)

submission.head()

Predicting: 100%|██████████| 54/54 [00:07<00:00,  7.55it/s]


,Id,Autor
0,7oYYGUufTgpKPCcVUhTpAS,Grigore Vieru
1,EGWuQxkFSjW59anvGxW7nS,Grigore Vieru
2,XfHVBFedR9cDb4DBvuwtsF,Grigore Vieru
3,E3EbfZuVsZhUxqRYvBbywM,Lucian Blaga
4,kAxEyKrjhpvUu25CAjh5F6,Vasile Alecsandri


In [17]:
submission.to_csv(f"{root_path}/submission.csv", index=False)